In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
from math import prod

import pandas as pd
from tqdm.auto import tqdm

from assignment_1.modules.models import fit_svd, predict_svd, fit_bias
from assignment_1.modules.util import calc_mse, load_csv, random_split

DATA_DIR = "../data"  # notebook runs from notebooks/

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
TASK_TRAIN_FILE = "tasks_3_4_train.csv"
TASK_TEST_FILE = "task_3_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)

In [11]:
train.head(4)

,user,item,timestamp,rating
0,u1838647363,i0846907466,1360914337,3
1,u7093796076,i2832259763,1472021070,4
2,u4779464116,i9244553928,1480231213,5
3,u2107634551,i2346931943,1343026318,4


In [12]:
for name, df in [("train", train), ("test", test)]:
    t = pd.to_datetime(df.timestamp, unit="s")
    odd = (t < "2005-01-01") | (t > "2018-12-31")
    print(f"{name}: {odd.sum():,} rows outside 2005–2018 ({odd.mean():.3%})")

train: 1,944 rows outside 2005–2018 (0.136%)
test: 30 rows outside 2005–2018 (0.150%)


In [17]:
tr, val = random_split(train, val_frac=0.25, seed=0)

SCALE = (tr.rating.min(), tr.rating.max())


In [18]:
def val_svd_mse(**params):
    model = fit_svd(tr, rating_scale=SCALE, **params)
    return calc_mse(val.rating, predict_svd(model, val.user, val.item))

base = val_svd_mse(n_factors=1, n_epochs=50)

print(f"baseline val svd MSE = {base:.4f}  ->  2% target ≈ {base * 0.98:.4f}")

baseline val svd MSE = 0.9428  ->  2% target ≈ 0.9240


In [21]:
def val_bias_mse(**params):
    model = fit_bias(tr, rating_scale=SCALE, **params)
    return calc_mse(val.rating, predict_svd(model, val.user, val.item))

bias = val_bias_mse(reg_u=1, reg_i=5, n_epochs=25)
print(f"bias val MSE = {bias:.4f}  ({(1 - bias / base) * 100:+.1f}% vs baseline)")


bias val MSE = 0.9243  (+2.0% vs baseline)


In [22]:
grid = {"reg_u": [0.3, 1, 3, 5], "reg_i": [2, 5, 10, 20], "n_epochs": [25]}

rows = []
for values in tqdm(itertools.product(*grid.values()), total=prod(map(len, grid.values()))):
    params = dict(zip(grid, values))
    s = val_bias_mse(**params)
    rows.append({**params, "val_mse": s, "vs_base_%": (1 - s / base) * 100})

results = pd.DataFrame(rows).sort_values("val_mse")
results.head(10)

  0%|          | 0/16 [00:00<?, ?it/s]

,reg_u,reg_i,n_epochs,val_mse,vs_base_%
9,3.0,5,25,0.922915,2.112132
10,3.0,10,25,0.923832,2.014892
8,3.0,2,25,0.923868,2.011085
5,1.0,5,25,0.924256,1.969933
13,5.0,5,25,0.924497,1.944373
6,1.0,10,25,0.925189,1.870923
4,1.0,2,25,0.925206,1.869148
14,5.0,10,25,0.925382,1.850490
12,5.0,2,25,0.925477,1.840413
1,0.3,5,25,0.926133,1.770841
